[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/deployment.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239303-lesson-8-deployment)

# Introducción al despliegue

## Repaso

Construimos hasta llegar a un agente con memoria:

* `actuar`: dejar que el modelo llame a herramientas específicas
* `observar`: pasarle al modelo la salida de la herramienta
* `razonar`: dejar que el modelo razone sobre la salida de la herramienta para decidir qué hacer después (por ejemplo, llamar a otra herramienta o simplemente responder directamente)
* `persistir el estado`: usar un *checkpointer* en memoria para permitir conversaciones largas con interrupciones

## Objetivos

Ahora veremos cómo desplegar nuestro agente de forma local en Studio y en `LangGraph Cloud`.

In [3]:
%%capture --no-stderr
%pip install --quiet -U langgraph_sdk langchain_core

## Conceptos

Hay algunos conceptos centrales que conviene entender:

`LangGraph` —
- Biblioteca de Python y JavaScript
- Permite crear flujos de trabajo con agentes

`LangGraph API` —
- Empaqueta el código del grafo
- Ofrece una cola de tareas para gestionar las operaciones asíncronas
- Ofrece persistencia para mantener el estado entre interacciones

`LangSmith Deployment` (antes `LangGraph Cloud`) --
- Servicio alojado para la LangGraph API
- Permite desplegar grafos desde repositorios de GitHub
- También ofrece monitoreo y tracing de los grafos desplegados
- Se accede mediante una URL única para cada despliegue

`LangSmith Studio` (antes `LangGraph Studio`) --
- Entorno de desarrollo integrado (IDE) para aplicaciones con LangGraph
- Usa la API como back-end, lo que permite probar y explorar grafos en tiempo real
- Se puede ejecutar de forma local o con un despliegue en la nube. Ver más abajo.

`LangGraph SDK` --
- Biblioteca de Python para interactuar con los grafos de LangGraph desde código
- Ofrece una interfaz consistente para trabajar con grafos, ya estén servidos de forma local o en la nube
- Permite crear clientes, acceder a los *assistants* (configuraciones guardadas de un grafo), gestionar hilos y ejecutar *runs* (ejecuciones)

## Pruebas en local

## Studio

**⚠️ Aviso**

Desde que se filmaron estos videos, actualizamos Studio para que ahora se pueda ejecutar de forma local y abrir desde el navegador. Esta es la forma recomendada de usar Studio, en lugar de la aplicación de escritorio que se muestra en el video. Ahora se llama _LangSmith Studio_ y ya no _LangGraph Studio_. Las instrucciones de instalación detalladas están en la guía "Puesta a punto" que está al comienzo del curso. Puedes ver una descripción de Studio [aquí](https://docs.langchain.com/langsmith/studio) y los detalles del despliegue local [aquí](https://docs.langchain.com/langsmith/quick-start-studio#local-development-server).
Para iniciar el servidor de desarrollo local, ejecuta el siguiente comando en tu terminal, dentro del directorio `/studio` de este módulo:

```
langgraph dev
```

Deberías ver la siguiente salida:
```
- 🚀 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs
```

Abre tu navegador y entra a la URL de la Studio UI que aparece arriba.

In [1]:
if 'google.colab' in str(get_ipython()):
    raise Exception("Lamentablemente, LangGraph Studio por ahora no es compatible con Google Colab")

In [2]:
from langgraph_sdk import get_client

In [3]:
# Esta es la URL del servidor de desarrollo local
URL = "http://127.0.0.1:2024"
cliente = get_client(url=URL)

# Buscar todos los grafos alojados
asistentes = await cliente.assistants.search()

In [4]:
asistentes[-3]

{'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca',
 'graph_id': 'agent',
 'config': {},
 'metadata': {'created_by': 'system'},
 'name': 'agent',
 'created_at': '2025-03-04T22:57:28.424565+00:00',
 'updated_at': '2025-03-04T22:57:28.424565+00:00',
 'version': 1}

In [3]:
# Creamos un hilo para llevar el seguimiento del estado de nuestra ejecución
hilo = await cliente.threads.create()

Ahora podemos ejecutar nuestro agente [con `cliente.runs.stream`](https://docs.langchain.com/oss/python/langgraph/graph-api/#stream-and-astream) usando:

* El `thread_id`
* El `graph_id`
* El `input`
* El `stream_mode`

Hablaremos del streaming en detalle en un módulo futuro.

Por ahora, basta con saber que estamos haciendo [streaming](https://docs.langchain.com/langsmith/streaming) del valor completo del estado después de cada paso del grafo con `stream_mode="values"`.

El estado queda en `chunk.data`.

In [4]:
from langchain_core.messages import HumanMessage

# Entrada
entrada = {"messages": [HumanMessage(content="Multiplica 3 por 2.")]}

# Streaming
async for chunk in cliente.runs.stream(
        hilo['thread_id'],
        "agente",
        input=entrada,
        stream_mode="values",
    ):
    if chunk.data and chunk.event != "metadata":
        print(chunk.data['messages'][-1])

{'content': 'Multiply 3 by 2.', 'additional_kwargs': {'example': False, 'additional_kwargs': {}, 'response_metadata': {}}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': 'cdbd7bd8-c476-4ad4-8ab7-4ad9e3654267', 'example': False}
{'content': '', 'additional_kwargs': {'tool_calls': [{'index': 0, 'id': 'call_iIPryzZZxRtXozwwhVtFObNO', 'function': {'arguments': '{"a":3,"b":2}', 'name': 'multiply'}, 'type': 'function'}]}, 'response_metadata': {'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_157b3831f5'}, 'type': 'ai', 'name': None, 'id': 'run-06c7243c-426d-4c81-a113-f1335dda5fb2', 'example': False, 'tool_calls': [{'name': 'multiply', 'args': {'a': 3, 'b': 2}, 'id': 'call_iIPryzZZxRtXozwwhVtFObNO', 'type': 'tool_call'}], 'invalid_tool_calls': [], 'usage_metadata': None}
{'content': '6', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'tool', 'name': 'multiply', 'id': '988cb170-f6e6-43c1-82fd-309f519abe6d', 'tool_call_id': 'c

## Pruebas en la nube

Podemos desplegar en la nube con LangSmith, como se explica [aquí](https://docs.langchain.com/langsmith/deployment-quickstart#deploy-from-github-with-langgraph-cloud).

### Crea un repositorio nuevo en GitHub

* Entra a tu cuenta de GitHub
* Haz clic en el ícono "+" de la esquina superior derecha y elige `"New repository"`
* Ponle un nombre a tu repositorio (por ejemplo, `langchain-academy`)

### Agrega tu repositorio de GitHub como remoto

* Vuelve a la terminal donde clonaste `langchain-academy` al comienzo del curso
* Agrega tu repositorio de GitHub recién creado como remoto

```
git remote add origin https://github.com/your-username/your-repo-name.git
```
* Haz push a él
```
git push -u origin main
```

### Conecta LangSmith a tu repositorio de GitHub

* Entra a [LangSmith](hhttps://smith.langchain.com/)
* Haz clic en la pestaña `deployments` del panel izquierdo de LangSmith
* Agrega `+ New Deployment`
* Luego, elige el repositorio de GitHub (por ejemplo, `langchain-academy`) que acabas de crear para el curso
* Apunta el `LangGraph API config file` a uno de los directorios `studio`
* Por ejemplo, para el módulo 1 usa: `module-1/studio/langgraph.json`
* Define tus claves de API (por ejemplo, puedes copiarlas de tu archivo `module-1/studio/.env`)

![Captura de pantalla de la configuración de un nuevo despliegue en LangSmith](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbad4fd61c93d48e5d0f47_deployment2.png)

### Trabaja con tu despliegue

Después podemos interactuar con nuestro despliegue de varias formas:

* Con el SDK, como antes.
* Con [LangGraph Studio](https://docs.langchain.com/langsmith/deployment-quickstart#3-test-your-application-in-studio).

![Captura de pantalla de un despliegue abierto en LangGraph Studio](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbad4fa159a09a51d601de_deployment3.png)

Para usar el SDK aquí en el notebook, ¡solo asegúrate de que `LANGSMITH_API_KEY` esté definida!

In [1]:
import os, getpass

def _definir_variable(variable: str):
    if not os.environ.get(variable):
        os.environ[variable] = getpass.getpass(f"{variable}: ")

_definir_variable("LANGSMITH_API_KEY")

In [ ]:
# Reemplaza esto con la URL de tu grafo desplegado
URL = "https://langchain-academy-8011c561878d50b1883f7ed11b32d720.default.us.langgraph.app"
cliente = get_client(url=URL)

# Buscar todos los grafos alojados
asistentes = await cliente.assistants.search()

In [37]:
# Elegir el agente
agente = asistentes[0]

In [38]:
agente

{'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca',
 'graph_id': 'agent',
 'created_at': '2024-08-23T17:58:02.722920+00:00',
 'updated_at': '2024-08-23T17:58:02.722920+00:00',
 'config': {},
 'metadata': {'created_by': 'system'}}

In [40]:
from langchain_core.messages import HumanMessage

# Creamos un hilo para llevar el seguimiento del estado de nuestra ejecución
hilo = await cliente.threads.create()

# Entrada
entrada = {"messages": [HumanMessage(content="Multiplica 3 por 2.")]}

# Streaming
async for chunk in cliente.runs.stream(
        hilo['thread_id'],
        "agente",
        input=entrada,
        stream_mode="values",
    ):
    if chunk.data and chunk.event != "metadata":
        print(chunk.data['messages'][-1])

{'content': 'Multiply 3 by 2.', 'additional_kwargs': {'example': False, 'additional_kwargs': {}, 'response_metadata': {}}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': '8ea04559-f7d4-4c82-89d9-c60fb0502f21', 'example': False}
{'content': '', 'additional_kwargs': {'tool_calls': [{'index': 0, 'id': 'call_EQoolxFaaSVU8HrTnCmffLk7', 'function': {'arguments': '{"a":3,"b":2}', 'name': 'multiply'}, 'type': 'function'}]}, 'response_metadata': {'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_3aa7262c27'}, 'type': 'ai', 'name': None, 'id': 'run-b0ea5ddd-e9ba-4242-bb8c-80eb52466c76', 'example': False, 'tool_calls': [{'name': 'multiply', 'args': {'a': 3, 'b': 2}, 'id': 'call_EQoolxFaaSVU8HrTnCmffLk7', 'type': 'tool_call'}], 'invalid_tool_calls': [], 'usage_metadata': None}
{'content': '6', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'tool', 'name': 'multiply', 'id': '1bf558e7-79ef-4f21-bb66-acafbd04677a', 'tool_call_id': 'c